In [2]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown

In [3]:
load_dotenv(override=True)

True

In [4]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [5]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

As of April 2026, several AI agent frameworks have emerged, each catering to specific use cases:

- **LangGraph**: Ideal for complex, stateful workflows, LangGraph models agent behavior as directed graphs, supporting single, multi, and hierarchical-agent patterns. ([workday.com](https://www.workday.com/en-us/perspectives/ai/2026/01/top-ai-agent-frameworks.html?utm_source=openai))

- **CrewAI**: Designed for role-based agent teams, CrewAI facilitates collaboration among specialized agents, making it suitable for research teams and content generation. ([leaper.dev](https://leaper.dev/blog/ai-agent-frameworks-2026?utm_source=openai))

- **AutoGen**: Developed by Microsoft Research, AutoGen focuses on conversational multi-agent systems, enabling agents to engage in dialogues and coordinate tasks. ([delx.ai](https://delx.ai/agents/best-ai-agent-frameworks-2026?utm_source=openai))

- **OpenClaw**: Emphasizing interactive assistants and team integration, OpenClaw supports channel-native communication and a flexible skill system, allowing agents to operate across multiple devices. ([openclawsetup.dev](https://openclawsetup.dev/blog/best-ai-agent-frameworks-2026?utm_source=openai))

- **Semantic Kernel**: Backed by Microsoft, Semantic Kernel is tailored for enterprise .NET/Java agents, integrating seamlessly with existing enterprise systems. ([delx.ai](https://delx.ai/agents/best-ai-agent-frameworks-2026?utm_source=openai))

The AI agent framework ecosystem is consolidating around these major players, with the OpenAI Agents SDK and LangGraph leading in adoption. The emergence of the Multi-Agent Coordination Protocol (MCP) has become an interoperability standard, further streamlining development across different frameworks. ([agentscout.live](https://agentscout.live/tech/ai-agents/insight/ai-agent-framework-consolidation-developer-preference-shifts/?utm_source=openai))

In addition to these frameworks, companies like Salesforce and Nvidia are advancing AI agent research. Salesforce's AI Foundry focuses on developing innovative AI tools for enterprise applications, while Nvidia's Nemotron Coalition aims to co-develop open frontier models, enhancing interoperability in the AI landscape. ([itpro.com](https://www.itpro.com/technology/artificial-intelligence/salesforce-ramps-up-agentic-ai-research-with-new-foundry-project?utm_source=openai))


## Highlights:
- [Salesforce ramps up agentic AI research with new foundry project](https://www.itpro.com/technology/artificial-intelligence/salesforce-ramps-up-agentic-ai-research-with-new-foundry-project?utm_source=openai), Published on Friday, March 27
- [Nvidia's Nemotron coalition brings eight AI labs together to build open frontier models](https://www.tomshardware.com/tech-industry/artificial-intelligence/nvidias-nemoclaw-coalition-brings-eight-ai-labs-together-to-build-open-frontier-models?utm_source=openai), Published on Monday, March 16
- [Nvidia targets open source interoperability with new model coalitions, agentic frameworks](https://www.itpro.com/technology/artificial-intelligence/nvidia-targets-open-source-interoperability-with-new-model-coalitions-agentic-frameworks?utm_source=openai), Published on Wednesday, March 18 

In [16]:

HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [8]:
message = "Latest AI Agent frameworks in 2026"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find recent developments in AI frameworks that are specifically designed for agent-based systems.', query='latest AI agent frameworks 2026'), WebSearchItem(reason='To understand current trending technologies and tools that support AI agent development and deployment.', query='AI agent technology trends 2026'), WebSearchItem(reason='To gather insights on industry leaders and new entrants in the AI agent framework space in 2026.', query='top AI agent frameworks companies 2026')]


In [9]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(os.environ.get('FROM_EMAIL'))
    to_email = To(os.environ.get('TO_EMAIL'))
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [10]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [11]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [12]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [13]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [15]:
query ="Latest AI Agent frameworks in 2026."

with trace("Research trace"):
  print("Starting research....")
  
  search_plan = await plan_searches(query)
  search_results = await perform_searches(search_plan)
  report = await write_report(query, search_results)
  await send_email(report)
  print("Hooray, all done!")

Starting research....
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray, all done!
